#Honours Project - Practical Demonstrator
## Student-Supervisor Pairing via Keyword Extraction

#Sam Wilson-Perkins, 40589154

You will need the testfile.csv and acm-codes.html files, with the latter needing to be placed into the content folder.

#Installers/Dependancies

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

import csv
import os
import pandas as pd
from bs4 import BeautifulSoup
import spacy
import re
from collections import defaultdict
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

dir_path = "/"
test_name = "testfile.csv"

#Opening CSV file

In [ ]:
test_path = os.path.join(dir_path, test_name)

with open(test_path, mode ='r', encoding="cp1252") as csvFile:
  df = pd.read_csv(csvFile)
  print(df.loc[0])

#Setup spaCy and build ACM Taxonomy

For my keyword extraction, I'm using 4 strategies to pull these out, plus an aggregated collection of the 4.
These strategies are:

*   "Exact", which matches exact phrases from the text body.
*   "Word", which matches individual words from the text.
*   "Lemma", which pulls words based upon their lemma (i.e. "changes" or "changing" being abstracted down to "change").
*   "Related", which searches for related concepts and phrases from the text.

These, along with the aggregated strategy, find a good selection of keywords from the text body.





In [ ]:
#Load spaCy model
nlp = spacy.load("en_core_web_sm")

def extract_acm_taxonomy(soup):
    """Extract ACM taxonomy codes and names from HTML."""
    acm_taxonomy = {}

    #Find codes and keyword names from ACM webpage
    for li in soup.find_all('li', id=lambda x: x and x.startswith('code:')):
        code = li['id'].replace('code:', '')
        link = li.find('a', recursive=False)

        if link:
            category_name = ''
            for content in li.children:
                if isinstance(content, str):
                    category_name += content
                elif content.name == 'a':
                    continue
                else:
                    break

            category_name = category_name.strip()
            if category_name.startswith(':'):
                category_name = category_name[1:].strip()

            if category_name:
                acm_taxonomy[code] = {
                    'name': category_name,
                    'level': code.count('.')
                }

    return acm_taxonomy

def build_searchable_index(acm_taxonomy):
    """Create a searchable index with words and lemmas for each term."""
    searchable = {}

    #Get top-level categories
    top_level = {code: info['name'] for code, info in acm_taxonomy.items()
                 if len(code) <= 2 and code.endswith('.')}

    for code, info in acm_taxonomy.items():
        term = info['name'].lower()
        parent_code = code[0] + '.'
        parent_name = top_level.get(parent_code, 'Unknown')

        #Process with spaCy for lemmatization
        doc = nlp(term)
        lemmas = {token.lemma_.lower() for token in doc if not token.is_stop}

        searchable[term] = {
            'code': code,
            'category': parent_name,
            'original_name': info['name'],
            'words': set(term.split()),
            'lemmas': lemmas
        }

    return searchable

def expand_abbreviations(text):
    """Replace common abbreviations with full terms."""
    abbreviations = {
        'ai': 'artificial intelligence',
        'ml': 'machine learning',
        'dl': 'deep learning',
        'nn': 'neural network',
        'cnn': 'convolutional neural network',
        'rnn': 'recurrent neural network',
        'nlp': 'natural language processing',
        'cv': 'computer vision',
        'db': 'database',
        'dbms': 'database management system',
        'os': 'operating system',
        'api': 'application programming interface',
        'ui': 'user interface',
        'ux': 'user experience',
        'gui': 'graphical user interface',
        'oop': 'object oriented programming',
        'sql': 'structured query language',
        'html': 'hypertext markup language',
        'css': 'cascading style sheets',
        'http': 'hypertext transfer protocol',
        'tcp': 'transmission control protocol',
        'ip': 'internet protocol',
        'lan': 'local area network',
        'wan': 'wide area network',
        'ram': 'random access memory',
        'cpu': 'central processing unit',
        'gpu': 'graphics processing unit',
        'hci': 'human computer interaction',
        'ar': 'augmented reality',
        'vr': 'virtual reality',
        'iot': 'internet of things',
    }

    #Convert non-string inputs (like NaN) to empty string
    if not isinstance(text, str):
        text = str(text) if not pd.isna(text) else ''

    text_lower = text.lower()
    for abbrev, full_term in abbreviations.items():
        pattern = r'\b' + re.escape(abbrev) + r'\b'
        text_lower = re.sub(pattern, full_term, text_lower)

    return text_lower

def get_related_terms():
    """Map common words to related ACM concepts."""
    return {
        'camera': ['computer vision', 'image processing', 'visual recognition'],
        'video': ['computer vision', 'video processing', 'multimedia'],
        'image': ['computer vision', 'image processing', 'graphics'],
        'detect': ['pattern recognition', 'detection'],
        'recognize': ['pattern recognition', 'recognition'],
        'classify': ['classification', 'pattern recognition'],
        'predict': ['prediction', 'machine learning'],
        'robot': ['robotics', 'autonomous systems'],
        'autonomous': ['robotics', 'autonomous systems'],
        'network': ['networking', 'computer networks'],
        'web': ['web applications', 'world wide web'],
        'mobile': ['mobile computing', 'mobile applications'],
        'app': ['application', 'software'],
        'code': ['programming', 'software development'],
        'program': ['programming', 'software development'],
        'algorithm': ['algorithms', 'computational methods'],
        'data': ['data management', 'data processing'],
        'security': ['computer security', 'information security'],
        'encrypt': ['encryption', 'cryptography'],
        'neural': ['neural networks', 'machine learning'],
        'learn': ['machine learning', 'learning systems'],
        'train': ['machine learning', 'training'],
        'model': ['modeling', 'machine learning'],
    }

def extract_keywords(text, searchable_terms, max_keywords=10, sort_by='score'):
    """
    Extract ACM keywords from text using multiple matching strategies.

    Strategies:
    1. Exact phrase matching
    2. Individual word matching
    3. Lemmatized word matching (handles word variations)
    4. Related concept matching

    Args:
        text: Input text to extract keywords from
        searchable_terms: Pre-built searchable index
        max_keywords: Maximum number of keywords to return
        sort_by: 'score' (default) or 'code_length'

    Returns:
        Dictionary with keys: 'aggregated', 'exact', 'word', 'lemma', 'related'
    """
    text_expanded = expand_abbreviations(text)
    related_terms = get_related_terms()

    #Process text with spaCy
    doc = nlp(text_expanded)
    text_lemmas = {token.lemma_.lower() for token in doc
                   if not token.is_stop and token.is_alpha}
    text_words = {token.text.lower() for token in doc
                  if not token.is_stop and token.is_alpha}

    #Store matches by strategy
    matches_by_strategy = {
        'exact': [],
        'word': [],
        'lemma': [],
        'related': []
    }

    #Strategy 1: Exact phrase matching
    for term, info in searchable_terms.items():
        if term in text_expanded:
            matches_by_strategy['exact'].append({
                'keyword': info['original_name'],
                'category': info['category'],
                'code': info['code'],
                'score': len(term.split()) * 10,
                'match_type': 'exact'
            })

    #Strategy 2: Word matching
    for term, info in searchable_terms.items():
        word_overlap = len(info['words'] & text_words)
        if word_overlap > 0:
            score = (word_overlap / len(info['words'])) * 5
            matches_by_strategy['word'].append({
                'keyword': info['original_name'],
                'category': info['category'],
                'code': info['code'],
                'score': score,
                'match_type': 'word'
            })

    #Strategy 3: Lemma matching
    for term, info in searchable_terms.items():
        if info['lemmas']:
            lemma_overlap = len(info['lemmas'] & text_lemmas)
            if lemma_overlap > 0:
                score = (lemma_overlap / len(info['lemmas'])) * 7
                matches_by_strategy['lemma'].append({
                    'keyword': info['original_name'],
                    'category': info['category'],
                    'code': info['code'],
                    'score': score,
                    'match_type': 'lemma'
                })

    #Strategy 4: Related terms
    for word in text_words:
        if word in related_terms:
            for related in related_terms[word]:
                for term, info in searchable_terms.items():
                    if related in term:
                        matches_by_strategy['related'].append({
                            'keyword': info['original_name'],
                            'category': info['category'],
                            'code': info['code'],
                            'score': 3,
                            'match_type': 'related'
                        })

    #Process individual strategies
    results_by_strategy = {}
    for strategy_name, matches in matches_by_strategy.items():
        #Aggregate duplicates within each strategy
        aggregated = defaultdict(lambda: {'score': 0})
        for match in matches:
            key = (match['keyword'], match['code'])
            aggregated[key]['score'] += match['score']
            aggregated[key]['keyword'] = match['keyword']
            aggregated[key]['category'] = match['category']
            aggregated[key]['code'] = match['code']

        #Convert to list
        strategy_results = []
        for data in aggregated.values():
            strategy_results.append({
                'keyword': data['keyword'],
                'category': data['category'],
                'code': data['code'],
                'score': data['score'],
                'match_types': strategy_name
            })

        #Sort based on preference
        if sort_by == 'code_length':
            strategy_results.sort(key=lambda x: (-len(x['code']), -x['score']))
        else:
            strategy_results.sort(key=lambda x: (-x['score'], -len(x['code'])))

        results_by_strategy[strategy_name] = strategy_results[:max_keywords]

    #Create aggregated results (combining all strategies)
    all_matches = []
    for matches in matches_by_strategy.values():
        all_matches.extend(matches)

    #Aggregate duplicate matches across all strategies
    aggregated = defaultdict(lambda: {'score': 0, 'match_types': set()})
    for match in all_matches:
        key = (match['keyword'], match['code'])
        aggregated[key]['score'] += match['score']
        aggregated[key]['match_types'].add(match['match_type'])
        aggregated[key]['keyword'] = match['keyword']
        aggregated[key]['category'] = match['category']
        aggregated[key]['code'] = match['code']

    #Convert to list
    aggregated_results = []
    for data in aggregated.values():
        aggregated_results.append({
            'keyword': data['keyword'],
            'category': data['category'],
            'code': data['code'],
            'score': data['score'],
            'match_types': ', '.join(sorted(data['match_types']))
        })

    #Sort based on preference
    if sort_by == 'code_length':
        aggregated_results.sort(key=lambda x: (-len(x['code']), -x['score']))
    else:
        aggregated_results.sort(key=lambda x: (-x['score'], -len(x['code'])))

    results_by_strategy['aggregated'] = aggregated_results[:max_keywords]

    return results_by_strategy

#Load and parse HTML
with open('acm-codes.html', 'r', encoding='utf-8') as f:
    soup = BeautifulSoup(f.read(), 'html.parser')

#Build taxonomy and searchable index
acm_taxonomy = extract_acm_taxonomy(soup)
searchable_terms = build_searchable_index(acm_taxonomy)

print(f"Loaded {len(acm_taxonomy)} ACM codes")
print(f"Built searchable index with {len(searchable_terms)} terms")
print("Scoring is based upon the aggregation of the different strategies used to gather the keywords\n")


#Extract Keywords from Student Profiles

In [ ]:
#Process supervisor profiles
student_count = 0
for i, row in df.loc[0:100, ["Students", "Student Profiles"]].iterrows(): #Modify the 0:N so that the second number is the amount of people in the table
    name = row["Students"]
    text = row["Student Profiles"]

    #Skip if the profile is missing/NaN
    if pd.isna(text) or not isinstance(text, str) or text.strip() == '':
        continue

    student_count += 1

    #Extract keywords (sorted by score, then code length)
    keywords_by_strategy = extract_keywords(text, searchable_terms, max_keywords=10, sort_by='score')

    #Only use aggregated strategy
    strategy = 'aggregated'
    strategy_title = 'Aggregated (All Strategies)'
    strategy_color = '#2196F3'

    #Create a figure with 1 subplot
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    fig.suptitle(f'ACM Keywords for Student {student_count} ', fontsize=18, fontweight='bold')

    ax.axis('tight')
    ax.axis('off')

    keywords = keywords_by_strategy[strategy]

    #Prepare table data
    table_data = []
    if keywords:
        for j, kw in enumerate(keywords, 1):
            table_data.append([
                f"{j}",
                kw['keyword'],
                kw['category'],
                kw['code'],
                f"{kw['score']:.2f}",
                kw['match_types']
            ])
    else:
        #Create a single row with "No keywords found" message
        table_data.append(['', 'No keywords found', '', '', '', ''])

    #Create table
    table = ax.table(cellText=table_data,
                    colLabels=['#', 'Keyword', 'Category', 'Code', 'Score', 'Match Strategy'],
                    cellLoc='left',
                    loc='center',
                    colWidths=[0.05, 0.25, 0.25, 0.15, 0.1, 0.2])

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)

    #Style header row
    for j in range(6):
        table[(0, j)].set_facecolor(strategy_color)
        table[(0, j)].set_text_props(weight='bold', color='white', fontsize=11)

    #Style data rows
    if keywords:
        #Alternate row colors for keywords
        for j in range(1, len(table_data) + 1):
            for k in range(6):
                if j % 2 == 0:
                    table[(j, k)].set_facecolor('#f0f0f0')
                else:
                    table[(j, k)].set_facecolor('white')
    else:
        #Style the "No keywords found" row
        for k in range(6):
            table[(1, k)].set_facecolor('#ffe6e6')
            if k == 1:  #Only the keyword column has text
                table[(1, k)].set_text_props(style='italic', color='#666666')

    ax.set_title(strategy_title, fontsize=13, fontweight='bold', pad=20, loc='left')

    plt.tight_layout()
    plt.show()

#Extract Keywords from Supervisor Profiles

In [ ]:
#Process supervisor profiles
supervisor_count = 0
for i, row in df.loc[0:100, ["Supervisors", "Supervisor Profiles"]].iterrows(): #Modify the 0:N so that the second number is the amount of people in the table
    name = row["Supervisors"]
    text = row["Supervisor Profiles"]

    #Skip if the profile is missing/NaN
    if pd.isna(text) or not isinstance(text, str) or text.strip() == '':
        continue

    supervisor_count += 1

    #Extract keywords (sorted by score, then code length)
    keywords_by_strategy = extract_keywords(text, searchable_terms, max_keywords=10, sort_by='score')

    #Only use aggregated strategy
    strategy = 'aggregated'
    strategy_title = 'Aggregated (All Strategies)'
    strategy_color = '#2196F3'

    #Create a figure with 1 subplot
    fig, ax = plt.subplots(1, 1, figsize=(14, 8))
    fig.suptitle(f'ACM Keywords for Supervisor {supervisor_count}', fontsize=18, fontweight='bold')

    ax.axis('tight')
    ax.axis('off')

    keywords = keywords_by_strategy[strategy]

    #Prepare table data
    table_data = []
    if keywords:
        for j, kw in enumerate(keywords, 1):
            table_data.append([
                f"{j}",
                kw['keyword'],
                kw['category'],
                kw['code'],
                f"{kw['score']:.2f}",
                kw['match_types']
            ])
    else:
        #Create a single row with "No keywords found" message
        table_data.append(['', 'No keywords found', '', '', '', ''])

    #Create table
    table = ax.table(cellText=table_data,
                    colLabels=['#', 'Keyword', 'Category', 'Code', 'Score', 'Match Strategy'],
                    cellLoc='left',
                    loc='center',
                    colWidths=[0.05, 0.25, 0.25, 0.15, 0.1, 0.2])

    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)

    #Style header row
    for j in range(6):
        table[(0, j)].set_facecolor(strategy_color)
        table[(0, j)].set_text_props(weight='bold', color='white', fontsize=11)

    #Style data rows
    if keywords:
        #Alternate row colors for keywords
        for j in range(1, len(table_data) + 1):
            for k in range(6):
                if j % 2 == 0:
                    table[(j, k)].set_facecolor('#f0f0f0')
                else:
                    table[(j, k)].set_facecolor('white')
    else:
        #Style the "No keywords found" row
        for k in range(6):
            table[(1, k)].set_facecolor('#ffe6e6')
            if k == 1:  #Only the keyword column has text
                table[(1, k)].set_text_props(style='italic', color='#666666')

    ax.set_title(strategy_title, fontsize=13, fontweight='bold', pad=20, loc='left')

    plt.tight_layout()
    plt.show()

#Ranking via Keyword Similarity and Keyword Scoring
Extracts the ACM keyword codes from both student and supervisor keywords
*   Similarity = intersection / union
*   Returns a score between 0 (no overlap) and 1 (perfect match)

Example: If a student has codes {A, B, C} and supervisor has {B, C, D}, the similarity is 2/4 = 0.5

In [ ]:
def compute_keyword_similarity(student_keywords, supervisor_keywords):
    """
    Compute similarity between student and supervisor keywords.
    Uses Jaccard similarity on the set of keyword codes.
    """
    if not student_keywords or not supervisor_keywords:
        return 0.0

    student_codes = set(kw['code'] for kw in student_keywords)
    supervisor_codes = set(kw['code'] for kw in supervisor_keywords)

    intersection = len(student_codes & supervisor_codes)
    union = len(student_codes | supervisor_codes)

    if union == 0:
        return 0.0

    return intersection / union

def compute_weighted_keyword_similarity(student_keywords, supervisor_keywords):
    """
    Compute weighted similarity between student and supervisor keywords.
    Considers both keyword overlap and scores.
    Actual metric used for matching.
    """
    if not student_keywords or not supervisor_keywords:
        return 0.0

    student_dict = {kw['code']: kw['score'] for kw in student_keywords}
    supervisor_dict = {kw['code']: kw['score'] for kw in supervisor_keywords}

    all_codes = set(student_dict.keys()) | set(supervisor_dict.keys())

    """
    If student has code D.4.8 with score 22 and supervisor has D.4.8 with score 15: contributes 22×15 = 330
    If a code exists in only one profile: contributes 0
    """
    dot_product = sum(student_dict.get(code, 0) * supervisor_dict.get(code, 0)
                      for code in all_codes)

    #Square root of sum of squared scores for each profile, like computing vector lengths in geometry.
    student_magnitude = sum(score ** 2 for score in student_dict.values()) ** 0.5
    supervisor_magnitude = sum(score ** 2 for score in supervisor_dict.values()) ** 0.5

    if student_magnitude == 0 or supervisor_magnitude == 0:
        return 0.0

    """
    This is essentially the cosine of the angle between two vectors
    Returns 0-1, where 1 means identical keyword profiles (same keywords with proportional scores)
    """
    return dot_product / (student_magnitude * supervisor_magnitude)

#Extract keywords for all students
student_keywords_dict = {}
student_count = 0

"""Loops through all rows in the Students columns.
Skips empty/missing profiles.
Increments counter to create labels: "Student 1", "Student 2", etc.
Calls extract_keywords() to get the top 10 ACM keywords using the aggregated strategy
"""

for i, row in df.loc[0:100, ["Students", "Student Profiles"]].iterrows(): #EDIT HERE FOR CHANGE IN STUDENT NUMBERS
    name = row["Students"]
    text = row["Student Profiles"]

    if pd.isna(text) or not isinstance(text, str) or text.strip() == '':
        continue

    student_count += 1
    student_label = f"Student {student_count}"

    keywords_by_strategy = extract_keywords(text, searchable_terms, max_keywords=10, sort_by='code_length')
    student_keywords_dict[student_label] = keywords_by_strategy['aggregated']

#Extract keywords for all supervisors
supervisor_keywords_dict = {}
supervisor_count = 0
for i, row in df.loc[0:100, ["Supervisors", "Supervisor Profiles"]].iterrows(): #EDIT HERE FOR CHANGE IN SUPERVISOR NUMBERS.
    name = row["Supervisors"]
    text = row["Supervisor Profiles"]

    if pd.isna(text) or not isinstance(text, str) or text.strip() == '':
        continue

    supervisor_count += 1
    supervisor_label = f"Supervisor {supervisor_count}"

    keywords_by_strategy = extract_keywords(text, searchable_terms, max_keywords=10, sort_by='score')
    supervisor_keywords_dict[supervisor_label] = keywords_by_strategy['aggregated']

print(f"Found {len(student_keywords_dict)} students with keywords")
print(f"Found {len(supervisor_keywords_dict)} supervisors with keywords\n")

"""
Similarity Matrix below:

For each student, create a sub-dictionary
For each supervisor, compute weighted similarity score
Stores the score in the nested structure

The weighted cosine similarity is better than simple Jaccard because it only considers if they share the keyword.
Weighted cosine considers if they share this keyword AND how important it is to the person.

A keyword with high scores for both indicates strong mutual interest, resulting in higher similarity.
"""

#Compute similarity matrix
similarity_matrix = {}
for student_name, student_kw in student_keywords_dict.items():
    similarity_matrix[student_name] = {}
    for supervisor_name, supervisor_kw in supervisor_keywords_dict.items():
        similarity = compute_weighted_keyword_similarity(student_kw, supervisor_kw)
        similarity_matrix[student_name][supervisor_name] = similarity

#Create ranking table for each student
student_rankings = {}
for student_name in student_keywords_dict.keys():
    ranked_supervisors = sorted(
        supervisor_keywords_dict.keys(),
        key=lambda sup: similarity_matrix[student_name][sup],
        reverse=True
    )
    student_rankings[student_name] = ranked_supervisors

#Display the ranking table using matplotlib
fig, ax = plt.subplots(figsize=(16, 10))
ax.axis('tight')
ax.axis('off')

#Prepare table data
table_data = []
max_supervisors = len(supervisor_keywords_dict)

#Sort student labels numerically
sorted_students = sorted(student_keywords_dict.keys(),
                        key=lambda x: int(x.split()[1]))

for rank in range(1, max_supervisors + 1):
    row = [f"Rank {rank}"]
    for student_name in sorted_students:
        if rank <= len(student_rankings[student_name]):
            supervisor = student_rankings[student_name][rank - 1]
            similarity = similarity_matrix[student_name][supervisor]
            row.append(f"{supervisor}\n({similarity:.3f})")
        else:
            row.append("")
    table_data.append(row)

#Create column labels
col_labels = ['Ranking'] + sorted_students

#Create table
table = ax.table(cellText=table_data,
                colLabels=col_labels,
                cellLoc='center',
                loc='center')

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 3)

#Style header row
num_cols = len(col_labels)
for j in range(num_cols):
    if j == 0:
        table[(0, j)].set_facecolor('#607D8B')
    else:
        table[(0, j)].set_facecolor('#4CAF50')
    table[(0, j)].set_text_props(weight='bold', color='white', fontsize=11)

#Style data rows
for i in range(1, len(table_data) + 1):
    for j in range(num_cols):
        if j == 0:
            table[(i, j)].set_facecolor('#E0E0E0')
            table[(i, j)].set_text_props(weight='bold')
        else:
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')
            else:
                table[(i, j)].set_facecolor('white')

plt.title('Supervisor Rankings for Each Student\n(Based on Keyword Similarity)',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

#Heatmap visualization
fig, ax = plt.subplots(figsize=(14, 8))

students = sorted_students
supervisors = sorted(supervisor_keywords_dict.keys(),
                    key=lambda x: int(x.split()[1]))
matrix_data = np.zeros((len(supervisors), len(students)))

for i, supervisor in enumerate(supervisors):
    for j, student in enumerate(students):
        matrix_data[i, j] = similarity_matrix[student][supervisor]

im = ax.imshow(matrix_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)

ax.set_xticks(np.arange(len(students)))
ax.set_yticks(np.arange(len(supervisors)))
ax.set_xticklabels(students, fontsize=10)
ax.set_yticklabels(supervisors, fontsize=10)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Similarity Score', rotation=270, labelpad=20, fontsize=11)

for i in range(len(supervisors)):
    for j in range(len(students)):
        text = ax.text(j, i, f'{matrix_data[i, j]:.3f}',
                      ha="center", va="center", color="black", fontsize=9)

ax.set_title('Student-Supervisor Keyword Similarity Matrix', fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Students', fontsize=12, fontweight='bold')
ax.set_ylabel('Supervisors', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

#Student-Supervisor Matching using Greedy Algorithm

1. Calculate a weighted score combining similarity and load balancing
2. Process student-supervisor pairs by weighted score
3. Dynamically adjust weights as supervisors fill up


Adjust balance_weight and re-run to see different results

Lower (0.1-0.2): Prioritize match quality over balance

Higher (0.4-0.5): Prioritize load balance over match quality

In [ ]:
def greedy_hybrid_matching(similarity_matrix, students, supervisors,
                           supervisor_capacity=None, balance_weight=0.3):
    """
    Assign students to supervisors using a hybrid greedy algorithm that balances
    both match quality and load distribution.
    """

    num_students = len(students)
    num_supervisors = len(supervisors)

    #Auto-calculate capacity to distribute evenly
    if supervisor_capacity is None:
        supervisor_capacity = int(np.ceil(num_students / num_supervisors))

    print(f"Matching {num_students} students to {num_supervisors} supervisors")
    print(f"Supervisor capacity: {supervisor_capacity} students each")
    print(f"Balance weight: {balance_weight} (0=quality only, 1=balance only)\n")

    #Track assignments
    assigned_students = set()
    supervisor_counts = {sup: 0 for sup in supervisors}
    assignments = []

    #Greedy assignment loop
    while len(assigned_students) < num_students:
        best_pair = None
        best_weighted_score = -float('inf')

        #For each unassigned student
        for student in students:
            if student in assigned_students:
                continue

            #For each supervisor with capacity
            for supervisor in supervisors:
                if supervisor_counts[supervisor] >= supervisor_capacity:
                    continue

                #Calculate weighted score combining similarity and load balance
                similarity = similarity_matrix[student][supervisor]

                #Load balancing factor (prefer less loaded supervisors)
                current_load = supervisor_counts[supervisor]
                load_factor = 1.0 - (current_load / supervisor_capacity)

                #Combined score: balance between match quality and load distribution
                weighted_score = (1 - balance_weight) * similarity + balance_weight * load_factor

                if weighted_score > best_weighted_score:
                    best_weighted_score = weighted_score
                    best_pair = (student, supervisor, similarity)

        if best_pair is None:
            print(f"WARNING: Could not assign all students. {num_students - len(assigned_students)} remain.")
            break

        student, supervisor, similarity = best_pair

        #Calculate where this supervisor ranks in student's preferences
        student_prefs = sorted(
            [(sup, similarity_matrix[student][sup]) for sup in supervisors],
            key=lambda x: x[1],
            reverse=True
        )
        student_rank = next(i+1 for i, (sup, _) in enumerate(student_prefs) if sup == supervisor)

        #Record assignment
        assignments.append({
            'Student': student,
            'Supervisor': supervisor,
            'Similarity_Score': similarity,
            'Student_Rank': student_rank
        })

        assigned_students.add(student)
        supervisor_counts[supervisor] += 1

    print(f"Successfully assigned {len(assignments)} students\n")

    return pd.DataFrame(assignments)

#Analysis and Metrics

def analyze_assignments(df):

    num_students = len(df)

    metrics = {
        #Match Quality Metrics
        'avg_similarity': df['Similarity_Score'].mean(),
        'min_similarity': df['Similarity_Score'].min(),
        'max_similarity': df['Similarity_Score'].max(),
        'std_similarity': df['Similarity_Score'].std(),

        #Student Preference Metrics
        'avg_student_rank': df['Student_Rank'].mean(),
        'students_got_rank1': (df['Student_Rank'] == 1).sum(),
        'students_got_rank1_pct': (df['Student_Rank'] == 1).sum() / num_students * 100,
        'students_got_top3': (df['Student_Rank'] <= 3).sum(),
        'students_got_top3_pct': (df['Student_Rank'] <= 3).sum() / num_students * 100,
    }

    #Load Balancing Metrics
    supervisor_loads = df['Supervisor'].value_counts()
    metrics['min_load'] = supervisor_loads.min()
    metrics['max_load'] = supervisor_loads.max()
    metrics['avg_load'] = supervisor_loads.mean()
    metrics['std_load'] = supervisor_loads.std()
    metrics['load_imbalance'] = metrics['max_load'] - metrics['min_load']
    metrics['num_supervisors'] = len(supervisor_loads)

    return metrics


def print_summary(df, metrics):
    """
    Print a formatted summary of the assignment results.
    """
    print("=" * 70)
    print("ASSIGNMENT SUMMARY")
    print("=" * 70)
    print()

    print("MATCH QUALITY:")
    print(f"  Average Similarity Score:  {metrics['avg_similarity']:.3f}")
    print(f"  Similarity Range:          {metrics['min_similarity']:.3f} - {metrics['max_similarity']:.3f}")
    print(f"  Standard Deviation:        {metrics['std_similarity']:.3f}")
    print()

    print("STUDENT PREFERENCES:")
    print(f"  Average Student Rank:      {metrics['avg_student_rank']:.2f}")
    print(f"  Got 1st Choice:            {metrics['students_got_rank1']} ({metrics['students_got_rank1_pct']:.1f}%)")
    print(f"  Got Top 3 Choice:          {metrics['students_got_top3']} ({metrics['students_got_top3_pct']:.1f}%) ")
    print()

    print("LOAD BALANCING:")
    print(f"  Students per Supervisor:   {metrics['min_load']} - {metrics['max_load']} (avg: {metrics['avg_load']:.1f})")
    print(f"  Load Imbalance:            {metrics['load_imbalance']} students")
    print(f"  Standard Deviation:        {metrics['std_load']:.2f}")
    print()

    print("SUPERVISOR ASSIGNMENTS:")
    supervisor_counts = df['Supervisor'].value_counts().sort_index()
    for supervisor, count in supervisor_counts.items():
        print(f"  {supervisor}: {count} students")
    print()


def visualize_assignments(df, metrics):
    """
    Create visualizations of the assignment results.
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    #Plot 1: Similarity Score Distribution
    ax = axes[0, 0]
    ax.hist(df['Similarity_Score'], bins=20, color='#4CAF50', alpha=0.7, edgecolor='black')
    ax.axvline(metrics['avg_similarity'], color='red', linestyle='--', linewidth=2,
               label=f"Mean: {metrics['avg_similarity']:.3f}")
    ax.set_xlabel('Similarity Score', fontsize=11)
    ax.set_ylabel('Number of Students', fontsize=11)
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.set_title('Distribution of Match Quality', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    #Plot 2: Load Distribution by Supervisor
    ax = axes[0, 1]
    supervisor_counts = df['Supervisor'].value_counts().sort_index()
    supervisors = supervisor_counts.index
    counts = supervisor_counts.values

    colors = ['#2196F3' if c == metrics['avg_load'] else '#FF9800' if c > metrics['avg_load'] else '#4CAF50'
              for c in counts]
    ax.bar(range(len(supervisors)), counts, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(metrics['avg_load'], color='red', linestyle='--', linewidth=2,
               label=f"Mean: {metrics['avg_load']:.1f}")
    ax.set_xlabel('Supervisor', fontsize=11)
    ax.set_ylabel('Number of Students', fontsize=11)
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.set_title('Load Distribution Across Supervisors', fontsize=13, fontweight='bold')
    ax.set_xticks(range(len(supervisors)))
    ax.set_xticklabels(supervisors, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    #Plot 3: Student Rank Distribution
    ax = axes[1, 0]
    rank_counts = df['Student_Rank'].value_counts().sort_index()
    ranks = rank_counts.index
    counts = rank_counts.values

    ax.bar(ranks, counts, color='#9C27B0', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Student Rank (1 = Best Choice)', fontsize=11)
    ax.set_ylabel('Number of Students', fontsize=11)
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(integer=True))
    ax.set_title('Student Preference Satisfaction', fontsize=13, fontweight='bold')
    ax.set_xticks(ranks)
    ax.grid(axis='y', alpha=0.3)

    #Add percentage labels on bars
    for rank, count in zip(ranks, counts):
        pct = count / len(df) * 100
        ax.text(rank, count, f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

    #Plot 4: Top Matches Heatmap Sample (first 20 students)
    ax = axes[1, 1]

    #Sample first 20 students for readability
    sample_df = df.head(20)
    students = sample_df['Student'].values
    supervisors_in_sample = sample_df['Supervisor'].unique()

    #Create a small heatmap
    data_text = "TOP 20 ASSIGNMENTS:\n\n"
    for idx, row in sample_df.iterrows():
        data_text += f"{row['Student']} → {row['Supervisor']}\n"
        data_text += f"  Score: {row['Similarity_Score']:.3f} (Rank {row['Student_Rank']})\n\n"

    ax.text(0.05, 0.95, data_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='#f0f0f0', alpha=0.8))
    ax.axis('off')
    ax.set_title('Sample Assignments (First 20 Students)', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.show()

def match_students_to_supervisors(similarity_matrix, student_keywords_dict,
                                 supervisor_keywords_dict, balance_weight=0.3,
                                 supervisor_capacity=None, visualize=True):

    #Get sorted lists of students and supervisors
    students = sorted(student_keywords_dict.keys(),
                     key=lambda x: int(x.split()[1]))
    supervisors = sorted(supervisor_keywords_dict.keys(),
                        key=lambda x: int(x.split()[1]))

    #Run the matching algorithm
    assignments_df = greedy_hybrid_matching(
        similarity_matrix,
        students,
        supervisors,
        supervisor_capacity=supervisor_capacity,
        balance_weight=balance_weight
    )

    #Analyze results
    metrics = analyze_assignments(assignments_df)

    #Print summary
    print_summary(assignments_df, metrics)

    #Visualize if requested
    if visualize:
        visualize_assignments(assignments_df, metrics)

    return assignments_df

#Match students to supervisors using recommended settings
assignments_df = match_students_to_supervisors(
    similarity_matrix=similarity_matrix,
    student_keywords_dict=student_keywords_dict,
    supervisor_keywords_dict=supervisor_keywords_dict,
    balance_weight=0.3,  #70% match quality, 30% load balance
    visualize=True
)

#Display the results
print("\nFINAL ASSIGNMENTS:")
print(assignments_df.to_string())

#Save to CSV
assignments_df.to_csv('student_supervisor_assignments.csv', index=False)
print("\nAssignments saved to 'student_supervisor_assignments.csv'")

#Optionally download CSV file to local Downloads folder
Un-comment the line in this cell to allow for download of the created CSV file.

In [ ]:
from google.colab import files

#files.download('student_supervisor_assignments.csv')